In [1]:
import os, sys, random, subprocess, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import time
from kaggle_secrets import UserSecretsClient

# 1. Install Dependencies
!pip install -q comet_ml torchsummary

from comet_ml import start
from comet_ml.integration.pytorch import log_model
from torch import nn
from torchsummary import summary
from torch.utils.tensorboard import SummaryWriter
import cv2
from PIL import Image
from tqdm.notebook import tqdm

# 2. Setup Comet ML (Optional - if using Secrets)
try:
    user_secrets = UserSecretsClient()
    COMET_API_KEY = user_secrets.get_secret("COMET_API_KEY")
    os.environ['COMET_API_KEY'] = COMET_API_KEY
except:
    print("Comet API Key not found in Secrets. Make sure to set it manually if needed.")

# 3. Setup Seed and Device
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 4. Setup Github Repository
REPO = "an2dl-challenges-25-26"
TARGET_FOLDER = "challenge2"
WORKING_DIR = "/kaggle/working" # Kaggle's writable directory

os.chdir(WORKING_DIR)

# Clone repo (Fresh clone logic)
if os.path.exists(REPO):
    shutil.rmtree(REPO)
!git clone https://github.com/asarraa/{REPO}.git

# Navigate into the subfolder
project_path = os.path.abspath(os.path.join(WORKING_DIR, REPO, TARGET_FOLDER))

# Add to Python Path
if project_path not in sys.path:
    sys.path.append(project_path)

# Change Directory
os.chdir(project_path)
print(f"Setup Github complete. CWD: {os.getcwd()}")
# --------------------------------------------------------------------------
# 5. SETUP DATA (COPY FROM INPUT -> WRITABLE LOCAL)
# --------------------------------------------------------------------------
import shutil

INPUT_DATA_DIR = Path('/kaggle/input') 
# We use /kaggle/temp because it is writable and usually faster than /kaggle/working
# If you need to download modified files later, you'll need to move them to /kaggle/working at the end.
WRITABLE_DATA_ROOT = Path('/kaggle/working/local_data')

# 1. Find the Source Data in Read-Only Input
# We search for the csv file to automatically locate the dataset root regardless of folder names
found_csv = list(INPUT_DATA_DIR.rglob('train_patches.csv'))

if not found_csv:
    # Debug: Print what exists to help you troubleshoot
    print("Could not find train_patches.csv. Listing available input folders:")
    for p in INPUT_DATA_DIR.iterdir():
        print(p)
    raise FileNotFoundError("ERROR: Could not find 'train_patches.csv' in /kaggle/input.")

# found_csv[0] is ".../train/train_patches.csv"
# .parent is ".../train"
# .parent.parent is the root dataset folder we want to copy
SOURCE_PATH = found_csv[0].parent.parent
print(f"Source data identified at: {SOURCE_PATH}")

# 2. Copy to Writable Directory
if WRITABLE_DATA_ROOT.exists():
    print(f"Writable folder {WRITABLE_DATA_ROOT} already exists. Skipping copy.")
else:
    print(f"Copying data to writable directory: {WRITABLE_DATA_ROOT}...")
    print("This may take 1-2 minutes depending on dataset size...")
    shutil.copytree(SOURCE_PATH, WRITABLE_DATA_ROOT)
    print("Copy complete!")

# --------------------------------------------------------------------------
# 6. DEFINE PATHS
# --------------------------------------------------------------------------

# Now BASE_PATH points to the writable copy
BASE_PATH = WRITABLE_DATA_ROOT

TRAIN_IMG_DIR = BASE_PATH / 'train/images'
TRAIN_CSV_PATH = BASE_PATH / 'train/train_patches.csv'

# Safety checks
assert TRAIN_IMG_DIR.exists(), f'Error: Images not found at {TRAIN_IMG_DIR}'
assert TRAIN_CSV_PATH.exists(), f'Error: CSV not found at {TRAIN_CSV_PATH}'

print(f'Using device: {device}')
print(f'DATASET READY AND WRITABLE AT: {BASE_PATH}')

2025-12-11 18:32:02.688350: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765477922.709269     434 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765477922.715912     434 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Comet API Key not found in Secrets. Make sure to set it manually if needed.
Cloning into 'an2dl-challenges-25-26'...
remote: Enumerating objects: 627, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 627 (delta 119), reused 105 (delta 56), pack-reused 455 (from 1)
Receiving objects: 100% (627/627), 16.80 MiB | 31.34 MiB/s, done.
Resolving deltas: 100% (427/427), done.
Setup Github complete. CWD: /kaggle/working/an2dl-challenges-25-26/challenge2
Source data identified at: /kaggle/input/preprocess-v1-weighted-zip/preprocess_v1_weighted
Writable folder /kaggle/working/local_data already exists. Skipping copy.
Using device: cuda
DATASET READY AND WRITABLE AT: /kaggle/working/local_data


## **Get Loaders**

In [2]:
import lazy_loaders

BATCH_SIZE = 128
ADD_MASK_CHANNEL = False

# Ensure BASE_PATH is defined from previous cell
train_loader, val_loader, input_shape = lazy_loaders.get_loaders(
    batch_size=BATCH_SIZE, 
    add_mask_channel=ADD_MASK_CHANNEL, 
    base_path=BASE_PATH
)
test_loader, _ = lazy_loaders.get_test_loaders(
    batch_size=BATCH_SIZE, 
    add_mask_channel=ADD_MASK_CHANNEL, 
    base_path=BASE_PATH
)

print(f"Train/Val loaded. Input Shape: {input_shape}")

Train samples: 12125, Val samples: 3032
Input shape: (3, 224, 224)
Test samples: 12477, Input shape: (3, 224, 224)
Train/Val loaded. Input Shape: (3, 224, 224)


## **Training**

In [3]:
from launch_training import start_training
# Ensure 'models' is importable. If imports fail, restart kernel and run Cell 1 only.
from models import EfficientNetModel 

MODEL_NAME = "CNN"
TRAINING_PARAMS = {
    'epochs': 1,
    'patience': 10,
}

# Note: Ensure you imported comet_ml at the top if you want logging
trained_model, history, exp_id = start_training(
    model_name=MODEL_NAME,
    training_params=TRAINING_PARAMS,
    train_loader=train_loader,
    val_loader=val_loader,
    data_input_shape=input_shape,
    local_data_path = BASE_PATH
)

[DEBUG] Device type: <class 'NoneType'>, Final device: cuda, CUDA available: True
Using GPU: Tesla T4
--- Starting CNN on cuda ---


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.


✓ Updated input_shape to: (3, 224, 224)
Starting CNN model training...
Training Configuration:
 epochs: 1
learning_rate: 0.001
patience: 10
l1_lambda: 0
l2_lambda: 0
verbose: 10
criterion_name: CrossEntropyLoss
optimizer_name: adamw
Model Configuration:
 input_shape: (3, 224, 224)
num_classes: 4
num_blocks: 2
convs_per_block: 1
use_stride: False
stride_value: 2
padding_size: 1
pool_size: 2
initial_channels: 32
channel_multiplier: 2
dropout_rate_classifier_head: 0.2


COMET INFO: Experiment is live on comet.com https://www.comet.com/asarraa/test/7024e766278c413281fd70688b855979



[DEBUG] About to instantiate model...
[DEBUG] Initializing CNN model with the following parameters:
input_shape: (3, 224, 224)
num_classes: 4
num_blocks: 2
convs_per_block: 1
use_stride: False
stride_value: 2
padding_size: 1
pool_size: 2
initial_channels: 32
channel_multiplier: 2
dropout_rate_classifier_head: 0.2
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 224, 224]             896
              ReLU-2         [-1, 32, 224, 224]               0
         MaxPool2d-3         [-1, 32, 112, 112]               0
   VanillaCNNBlock-4         [-1, 32, 112, 112]               0
            Conv2d-5         [-1, 64, 112, 112]          18,496
              ReLU-6         [-1, 64, 112, 112]               0
         MaxPool2d-7           [-1, 64, 56, 56]               0
   VanillaCNNBlock-8           [-1, 64, 56, 56]               0
           Flatten-9               [-1, 2007

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : CNN_20251211_183248
COMET INFO:     url                   : https://www.comet.com/asarraa/test/7024e766278c413281fd70688b855979
COMET INFO:   Metrics:
COMET INFO:     F1/Training     : 0.27900624677869
COMET INFO:     F1/Validation   : 0.18861638114183868
COMET INFO:     Loss/Training   : 1.4441882402479034
COMET INFO:     Loss/Validation : 1.3138124961651725
COMET INFO:     train_f1        : 0.27900624677869
COMET INFO:     train_loss      : 1.4441882402479034
COMET INFO:     val_f1          : 0.18861638114183868
COMET INFO:     val_loss        : 1.3138124961651725
COMET INFO:   Others:
COMET INFO:     Name                : CNN_20251211_183248
COMET

Epoch   1/1 | Train: Loss=1.4442, F1 Score=0.2790 | Val: Loss=1.3138, F1 Score=0.1886
Best model restored from epoch 0 with val_f1 0.1886
Model saved to: /kaggle/working/local_data/experiments/models/CNN_20251211_183248.pt
Registry updated: ID CNN_20251211_183248


## **Cell for kaggle inference fixing**

In [10]:
from pathlib import Path
import importlib
import inference  # Import the module so we can reload it later

# 1. Locate the file
# Using the path seen in your error trace
target_file = Path("/kaggle/working/an2dl-challenges-25-26/challenge2/inference.py")

if target_file.exists():
    # 2. Read the code
    content = target_file.read_text()
    
    # 3. Define the fix
    # We replace the strict load call with the permissive one
    old_line = 'ckpt = torch.load(path, map_location=map_location)'
    new_line = 'ckpt = torch.load(path, map_location=map_location, weights_only=False)'
    
    if old_line in content:
        new_content = content.replace(old_line, new_line)
        target_file.write_text(new_content)
        print("✅ Fixed inference.py: Added weights_only=False to torch.load")
        
        # 4. Force reload of the module so Python uses the new code
        importlib.reload(inference)
        from inference import make_inference
        print("✅ Module reloaded. You can now run the inference cell.")
        
    elif "weights_only=False" in content:
        print("ℹ️ File was already patched.")
    else:
        print("⚠️ Could not match the exact line code. Please check inference.py content.")
else:
    print(f"❌ Could not find file at {target_file}")

✅ Fixed inference.py: Added weights_only=False to torch.load
✅ Module reloaded. You can now run the inference cell.


## **Inference**

In [12]:
from inference import make_inference
import glob


MODEL_PATH = str(BASE_PATH) + "/experiments/models/" + exp_id + ".pt"
MODEL_ID = exp_id

if MODEL_PATH:
    print(f"Running inference using: {MODEL_PATH}")
    make_inference(
        loader=test_loader, 
        device=device, 
        input_shape=input_shape, 
        model_path=MODEL_PATH, 
        model_name=MODEL_NAME, 
        experiment_id=exp_id, 
        base_path=str(BASE_PATH)
    )

    output_csv = Path("/kaggle/working/submission.csv")
    # Move submission to root for easy download
    if (BASE_PATH / "submission.csv").exists():
        shutil.move(str(BASE_PATH / "submission.csv"), str(output_csv))
        print(f"Submission saved to {output_csv}")
else:
    print("No model path found for inference.")

Running inference using: /kaggle/working/local_data/experiments/models/CNN_20251211_183248.pt
[DEBUG] Initializing CNN model with the following parameters:
input_shape: (3, 224, 224)
num_classes: 4
num_blocks: 2
convs_per_block: 1
use_stride: False
stride_value: 2
padding_size: 1
pool_size: 2
initial_channels: 32
channel_multiplier: 2
dropout_rate_classifier_head: 0.2
[INFO] Saved submission to /kaggle/working/local_data/CNN_20251211_183248_submission
